# 02 — Tokenizer

In [1]:
import os
if not os.path.exists('MiniGPT'):
    !git clone https://github.com/userKk1/MiniGPT.git
%cd MiniGPT

Cloning into 'MiniGPT'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 46 (delta 18), reused 34 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 43.79 KiB | 3.98 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/MiniGPT


In [2]:
!python -m src.data

README.md: 100% 1.30k/1.30k [00:00<00:00, 3.34MB/s]
Resolving data files: 100% 54/54 [00:00<00:00, 20209.91it/s]
Collected 2068 files, 10.00 MB
Wrote /content/MiniGPT/data/raw/corpus_raw.txt (10.5 MB)


In [3]:
import sys
sys.path.append('.')

from config import cfg, DATA_RAW_DIR, DATA_PROCESSED_DIR
import json

corpus_path = DATA_RAW_DIR / 'corpus_raw.txt'
text = corpus_path.read_text(encoding='utf-8')
print(f'Corpus length: {len(text):,} characters')

Corpus length: 10,472,708 characters


## 1. Build the vocabulary

Every unique character in the corpus becomes one token.

In [4]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
print(''.join(chars))

Vocab size: 1723
	
 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~ £¥§©ª«¬±¶·¹º»¿ÀÁÂÄÅÇÈÉÊËÌÍÎÏÑÒÓÔÖÙÚÛÜßàáâãäåçèéêëìíîïñòóôõöøùúûüāĄąćČčēėęīİķļŁłńņōřśŠšūźżŽžƒơưЅАБВГДЕЗКЛМНОПРСТУЧШабвгдежзийклмнопрстуфхцчшщыьэюяդր؋إابتجدرسعقكلمރ৳ரூ฿ლ៛ễ–—‘’“”•… ※₀₂₡₤₦₨₩₪₫€₭₮₱₲₴₵ℕ№←↑→↓⇒∀≥≦≧⊥─┌┐└┘├♥❤　、。「」あいうかがきけこしすずせそぞただちつてでとどなにのはばぶまみむもよらりるれわをィイカクグジスセタチッテデトニネバパビプムメモュラリルワンー一上下不与丑且业丞並个中串为主乃义之乎也了事于云互亘亙些交亥亦产亨亮亲人仅今从仔代令以件任企伊伍会传伪伶似伽佃位住佑何作使侃來例侑供依俄俐保俠信俣修俱倖候倦倩倭值做停偲傭像儲允先兎兜入全公共关其具内冊册写冲冴准凌凛凜凧凪凰凱出击函分切列则创初删判利刪别到則削前劉办功加务动劫勁動勺勿匁包化匡卓单博卜卡卯印即卿厨厩去参叉及发取叡叢口只可台右叶号司各合名后吗吞否启吻吾员和品哉哨哩啄喋喧喬喰嘉嘗嘩噂噌器回围图圃國在圭地场址坊坐块坛坦型埴堯堰堵堺塙塞增壕壬处复多大天太失头夷夹奄奎套女好如始姥姪娃娩嬉子字存孜孟安宋完宏宕定实客宥寅密寓實寵对导封将尋小少尖尤尭尾局屑属層峨峻崚嵩嵯嶺巌巖工已巳巴巷巽布帐帖带帳常幌幡平并幻幽庄庇序库应底庚庵廟建廻廿开异式引弘弛張当录彗彦彪彬径得徠徽心必志快忽态怜性总恕恢息恰悉悌您情惇惚惟惣惹惺意慧憐戊戏成我或戟户所手才托执批找护择括拷拼持指按挺挽换捧据捲捷捺掉掠接控推掬揃描提插換搜摑摺撒撞撫播撰操擢收改放敗敦数數文斐料斡斧断斯新方於无日旭时昂昊昌明昏易是昴時晃晄晋晏晒晟晦晨智暂暉暢曙曝曳更最月有朋服朔期未末本机李杏杖杜束来杭杵杷构枇析果柊柏某柑柘柚查柴柾标栖栗栞核根格桂桐桔档桧桶梁梓梛梢梦梧梯梶检棲椀椋椛椰椿楊楓楕楚楠楢業楯榊榎榛槇構槌槍槙槻樋標樟模樫樺樽橘橙檀檎檜櫂櫓櫛權次欣欽歎止正此步殆残毅毕毘毬汀求汐汝汲沌沒沓没沫法注洛洲洵洸活流测浏浩浬淀淋淳淵

## 2. Build encode/decode mappings

`stoi` (string-to-int) and `itos` (int-to-string)

In [6]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}

def encode(s:str)->list[int]:
    return [stoi[c] for c in s]

def decode(ids:list[int])->str:
    return ''.join(itos[i] for i in ids)


## 3. Sanity check: round-trip

In [11]:
sample = text[:200]
encoded = encode(sample)
decoded = decode(encoded)

print('Original :', repr(sample[:80]))
print('Decoded  :', repr(decoded[:80]))
assert decoded == sample, 'Round-trip failed — tokenizer is not lossless!'
print('Round-trip OK')

Original : '#!/usr/bin/env python\n# vim: tabstop=4 shiftwidth=4 softtabstop=4\n#\n# Copyright '
Decoded  : '#!/usr/bin/env python\n# vim: tabstop=4 shiftwidth=4 softtabstop=4\n#\n# Copyright '
Round-trip OK


## 4. Encode the full corpus and split train/val

In [12]:
import numpy as np

ids = encode(text)
data_arr = np.array(ids, dtype=np.uint16)  # fine as long as vocab_size < 65536

n = len(data_arr)

In [15]:
split_idx = int(n * cfg.data.train_split)
train_ids = data_arr[:split_idx]
val_ids = data_arr[split_idx:]

print(f'train: {len(train_ids):,} tokens')
print(f'val:   {len(val_ids):,} tokens')

train: 9,425,437 tokens
val:   1,047,271 tokens
